# 10. Community Detection & Summarization Pipeline

Notebook ini menjalankan algoritma **Leiden** (via GDS) untuk mendeteksi komunitas di dalam Knowledge Graph, kemudian men-generate **ringkasan (summary) komunitas** menggunakan LLM.

Ringkasan ini sangat penting untuk mendukung fitur **Global Search** (mendapatkan insight tingkat makro) dan **DRIFT Search**.

In [2]:
import os
import sys

# Tambahkan root direktori ke path agar bisa import dari src
sys.path.append(os.path.dirname(os.getcwd()))

from src.graph.client import GraphClient
from src.config import settings
from src.community.detection import CommunityDetector
from src.community.summarization import CommunitySummarizer
from src.services.providers import get_llm, get_embeddings

import logging
logging.basicConfig(level=logging.INFO)

In [3]:
client = GraphClient(
    uri=settings.neo4j_uri,
    user=settings.neo4j_user,
    password=settings.neo4j_password
)

INFO:src.graph.client:Connected to Neo4j at bolt://localhost:7687


## Skenario Konfigurasi Graph

Kita menggunakan konfigurasi Pure GraphRAG berdasarkan schema baru.

In [4]:
# ==========================================
# SKENARIO 1: Menggunakan Data Hasil Pure GraphRAG (LLM Extraction)
# ==========================================

NODE_LABELS = [
    'SymptomPattern', 
    'ProblemCluster', 
    'RootCausePattern', 
    'ActionPattern', 
    'Part',
    'MachineModel'
]

RELATIONSHIP_TYPES = [
    'EXHIBITED', 
    'CAUSED_BY', 
    'RESOLVED_BY',
    'INVOLVES_PART',
    'BELONGS_TO'
]

## Step 1: Detect Communities via Leiden
*(Catatan: Membutuhkan plugin Neo4j GDS terinstall)*

In [5]:
detector = CommunityDetector(client)

# Inject label dinamis sesuai skenario yang dipilih di atas
success = detector.detect(
    node_labels=NODE_LABELS,
    relationship_types=RELATIONSHIP_TYPES
)

if success:
    print("✅ Community detection successful!")
else:
    print("❌ Community detection failed. Please check if GDS is installed.")

INFO:src.community.detection:Starting Community Detection using GDS Leiden...
INFO:src.community.detection:Projected graph: 31400 nodes, 93196 relationships.
INFO:src.community.detection:Leiden completed. Found 10251 communities.
INFO:src.community.detection:Modularities across levels: [0.4908024540488062, 0.6904749812460188, 0.7604042701165672]
INFO:src.community.detection:Building Hierarchical Community nodes up to 3 levels...
INFO:src.community.detection:Hierarchical Community hierarchy built.


✅ Community detection successful!


## Step 2: Summarize Communities via LLM
Proses ini akan mengirimkan entitas di tiap komunitas ke Ollama untuk dirangkum. Waktu eksekusi bergantung pada jumlah komunitas dan kecepatan Ollama.

In [21]:
llm = get_llm(temperature=0.2)
embedder = get_embeddings()

summarizer = CommunitySummarizer(client, llm, embedder)
summarizer.summarize_all()
print("✅ Community summarization completed!")

INFO:src.community.summarization:Starting Hierarchical Community Summarization...
INFO:src.community.summarization:Summarizing Level 0 communities...
INFO:src.community.summarization:All Level 0 communities already summarized.
INFO:src.community.summarization:Summarizing Level 1 communities...
INFO:src.community.summarization:All Level 1 communities already summarized.
INFO:src.community.summarization:Summarizing Level 2 communities...
INFO:src.community.summarization:Found 2550 Level 2 communities to summarize.
INFO:src.community.summarization:Summarizing Level 2 community 3768 (1 sub-communities)...
INFO:httpx:HTTP Request: POST https://utaedaddsnpaoai.cognitiveservices.azure.com/openai/deployments/SS-gpt-5.4-mini/chat/completions?api-version=2024-12-01-preview "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://utaedaddsnpaoai.cognitiveservices.azure.com/openai/deployments/SS-text-embedding-3-small-2/embeddings?api-version=2024-12-01-preview "HTTP/1.1 200 OK"
INFO:src.community.

APITimeoutError: Request timed out.

In [22]:
client.close()